# 00. Data Inventory — Dunnhumby Transaction

## Mục tiêu

> Xác định bảng có cấu trúc gì và một dòng đại diện cho cái gì.

Notebook này chỉ kiểm tra cấu trúc: shape, physical dtype, semantic role, required columns, entity/time metadata, key cardinality, candidate grain và preview.

In [ ]:
%load_ext autoreload
%autoreload 2

from dataclasses import asdict
from pathlib import Path

import polars as pl

from propensity.eda import build_table_inventory
from propensity.io import scan_csv
from propensity.schema.dunnhumby import (
    TRANSACTION_SCHEMA_OVERRIDES,
    TRANSACTION_SPEC,
)

pl.Config.set_tbl_rows(20)
pl.Config.set_tbl_cols(20)

## 1. Load và validate structural contract

In [ ]:
project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent

transaction_path = (
    project_root / "data" / "raw" / "Dunnhumby" / "transaction_data.csv"
)
transactions = scan_csv(
    transaction_path,
    schema_overrides=TRANSACTION_SCHEMA_OVERRIDES,
)
TRANSACTION_SPEC.validate(transactions)
transactions

`scan_csv()` chỉ tạo lazy query plan. Các phép collect cần thiết nằm trong từng inventory function, không nằm trong notebook.

## 2. Build inventory

In [ ]:
inventory = build_table_inventory(transactions, TRANSACTION_SPEC)

## 3. Table overview

In [ ]:
pl.DataFrame([asdict(inventory.overview)])

## 4. Column structure và semantic roles

`physical_dtype` mô tả cách Polars lưu dữ liệu. `semantic_role` mô tả ý nghĩa nghiệp vụ do `DatasetSpec` cung cấp.

In [ ]:
inventory.columns

## 5. Entity và ID cardinality

Time column không nằm trong bảng này vì `DAY` là time axis, không phải key.

In [ ]:
inventory.keys

## 6. Candidate grain

Candidate grain của transaction là `BASKET_ID + PRODUCT_ID`. Grain chỉ được chấp nhận nếu tổ hợp này xác định duy nhất từng dòng.

In [ ]:
inventory.grain

## 7. Preview

In [ ]:
inventory.preview

## 8. Nhận xét cấu trúc

- Entity của bảng là `household_key`.
- Time axis là `DAY`, được lưu dưới dạng ordinal integer với `time_grain=day_index`; notebook không chuyển nó thành calendar date.
- `BASKET_ID` và `PRODUCT_ID` có physical dtype numeric nhưng semantic role là ID.
- Candidate grain là `BASKET_ID + PRODUCT_ID`. Nếu grain report có `is_unique=True`, một dòng được diễn giải là **một product line trong một basket**. Nếu `is_unique=False`, cần xem lại candidate grain hoặc dữ liệu trùng trước khi chốt row meaning.